In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
metadata_df = metadata_df.reset_index()

metadata_df

,samples,sample_title,tissue
0,GSM8463271,OB_pat_LuC_1,lung cancer (squamous cell carcinoma)
1,GSM8463272,OB_pat_LuC_2,lung cancer (adenocarcinoma)
2,GSM8463273,OB_pat_LuC_3,lung cancer (adenocarcinoma)
3,GSM8463274,OB_pat_LuC_16,lung cancer (squamous cell carcinoma)
4,GSM8463275,OB_pat_LuC_17,lung cancer (adenocarcinoma)
...,...,...,...
56,GSM8463327,OB_pat_LuC_128,lung cancer (adenocarcinoma)
57,GSM8463328,OB_pat_LuC_126,lung cancer (squamous cell carcinoma)
58,GSM8463329,OB_pat_LuC_136,lung cancer (squamous cell carcinoma)
59,GSM8463330,OB_pat_LuC_123,lung cancer (adenocarcinoma)


In [2]:
import re

# supplementary data located here: https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE274975&format=file&file=GSE274975%5Fraw%5Fcounts%2Etsv%2Egz  GSE274975_raw_counts.tsv.gz

expression_data_df = pd.read_csv('https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE274975&format=file&file=GSE274975%5Fraw%5Fcounts%2Etsv%2Egz', sep='\t', compression='gzip')

def extract_luc_number(col):
    m = re.search(r"LuC[-_]?(\d+)", col, flags=re.IGNORECASE)
    return f"OB_pat_LuC_{m.group(1)}" if m else col

new_columns = [extract_luc_number(c) for c in expression_data_df.columns]

expression_data_df.columns = new_columns
expression_data_df = expression_data_df.rename(columns={'Unnamed: 0': 'genes'})
expression_data_df = expression_data_df.set_index('genes')
expression_data_df = expression_data_df.T
expression_data_df.index.name = 'sample_title'

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["EnsemblGeneID", "Symbol"]]
    .dropna(subset=["EnsemblGeneID", "Symbol"])
    .set_index("EnsemblGeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in expression_data_df.columns if c in gene_mapping]
expression_data_df = expression_data_df[mappable].rename(columns=gene_mapping)


df_combined = metadata_df.merge(
    expression_data_df,
    left_on='sample_title',
    right_on='sample_title',
    how='inner'
)

# Clinical data available here: https://pmc.ncbi.nlm.nih.gov/articles/instance/11669362/bin/Table1.xlsx
clinical_data_df = pd.read_excel('Table1.xlsx')
clinical_data_df.to_csv('Table1_clinical_data.csv', index=False)

df_combined = clinical_data_df.merge(
    df_combined,
    left_on='Sample_ID',
    right_on='sample_title',
    how='inner'
)

data_file_name = f'{GEO_ID}_original.csv'
df_combined = df_combined.rename(columns=gene_mapping)
df_combined.to_csv(data_file_name, index=False)
df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_9670/3966341425.py:20: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,Sample_ID,RECIST Response,Response status available,"PFS time, months",Primary_tumor._metastasis,Targeted therapeutics (next line),Age at biosampling,Gender,Tumor_site (biosampling),tissue_type,...,DIP2A,DIP2A-IT1,S100B,PRMT2,LOC102723713,LOC102723502,LOC102723532,MAFIP,LOC389831,LOC100288966
0,OB_pat_LuC_104,PD,1,6,primary,"Pembrolizumab, Niraparib",45.0,male,Lung,Primary solid tumor,...,1844,31,29,1318,0,0,0,1,48,0
1,OB_pat_LuC_105,PR,0,4,primary,"Pembrolizumab, Niraparib",70.0,male,Lung,Primary solid tumor,...,1097,29,192,787,0,0,0,0,21,0
2,OB_pat_LuC_107,PD,1,3,primary,"Nivolumab, Ipilimumab, Denosumab",60.0,male,Lung,Primary solid tumor,...,1001,22,8,731,0,0,0,0,20,0
3,OB_pat_LuC_108,PD,1,3,primary,Pembrolizumab; Niraparib,66.0,male,Lung,Primary solid tumor,...,1465,19,16,1134,0,0,0,1,19,0
4,OB_pat_LuC_111,PD,1,3,primary,Pembrolizumab,65.0,male,Lung,Primary solid tumor,...,2021,32,61,1364,0,0,0,0,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,OB_pat_LuC_128,PR,1,15,primary,Pembrolizumab,68.0,male,Lung,Primary solid tumor,...,1065,31,0,1053,0,0,0,0,1,0
57,OB_pat_LuC_85,PD,1,2,primary,Nivolumab,60.0,female,Lung,Primary solid tumor,...,2837,39,42,2027,0,0,0,0,110,0
58,OB_pat_LuC_92,PR,0,6,primary,Atezolizumab,68.0,male,Lung,Primary solid tumor,...,1691,47,6,1070,0,0,0,0,101,0
59,OB_pat_LuC_131,CR,0,26,primary,"Pembrolizumab, Paclitaxel",67.0,male,Lung,Primary solid tumor,...,1605,31,13,1268,0,0,0,0,13,0


# Generate description


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'Table1_clinical_data.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': clinical_data_df.shape[0],
    'clinical_features': clinical_data_df.reset_index().shape[1] + metadata_df.shape[1],
    'genes': expression_data_df.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
